# 데이터 수집과 결합·검증

**용도** — 넷플릭스 공식 Top 10 데이터(구형 CSV + 공식 갱신분 TSV)를 결합해 281주 분석 기준 테이블을 만들고, 주차·국가·작품 수를 코드 수준으로 검증한다.

**보고서 대응** — 보고서 01장 '사용한 데이터' 표 · 기초 EDA 요약(구 데이터 cp1252 인코딩 변환, 중복 주차 제거로 281주 × 94개국 결합)

**환경** — Python 3.15 / pandas 3.1.1 / scikit-learn 1.9.1 / LightGBM 4.7.0 / SHAP 0.54.1 (requirements.txt)

**실행 순서** — 00(개요) → 01(수집·결합) → 02(EDA) → 03(장르 보완) → 04(언어장벽) → 05(시장파워·미러) → 06(LightGBM) → 07(보정·SHAP) → 08(데뷔)

In [1]:
# 공통 설정 — 데이터는 원본 저장소에서 읽고, 출력물은 이 폴더에만 기록(원본 보호)
from pathlib import Path
import sys
ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")
NBOUT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\넷플릭스_분석_노트북\출력물")
NBOUT_F = NBOUT / "figures"
for _d in [NBOUT, NBOUT_F, NBOUT / "models", NBOUT / "data" / "processed", NBOUT / "data" / "interim"]:
    _d.mkdir(parents=True, exist_ok=True)
print("데이터 루트 =", ROOT)
print("출력 폴더  =", NBOUT)

데이터 루트 = C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset
출력 폴더  = C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\넷플릭스_분석_노트북\출력물


**원본 코드** — `scripts/15_netflix_data_refresh.py`

In [2]:
# -*- coding: utf-8 -*-
# ─────────────────────────────────────────────────────────────────────────────
# [용도] 15. 추가 데이터(2026-11-11 갱신분) 수집 절차 가이드 + 다운로드 파일 검증
#        수집 자체는 브라우저 다운로드(수동)이고, 본 스크립트는 절차 문서화 +
#        내려받은 파일의 정합성(기간·행수·작품 수)을 검증한다
# [환경] Python 3.15.2 (Windows) / pandas 3.1.1 — requirements.txt 참고
#        실행: python scripts/15_netflix_data_refresh.py
# [절차] 1) https://top10.netflix.com/downloads 접속
#        2) All Weeks Global / All Weeks Countries TSV 다운로드
#        3) 다운로드일을 접미사로 붙여 data/raw/에 저장
#           예: all-weeks-global_2026-11-11.tsv, all-weeks-countries_2026-11-11.tsv
#        4) 본 스크립트 실행 → 기존 구형 파일과 결합 가능한지 검증
# [경로] ROOT = 스크립트 위치 기준 프로젝트 루트 (scripts/ 상위)
# [재현] 다운로드 시점에 따라 사이트 데이터가 갱신되므로 완전 동일 수집은 보장되지
#        않음 — 커밋된 raw 파일(2026-11-11 수집본)을 그대로 쓰면 보고서와 동일
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")

def inspect(path, sep, **kw):
    df = pd.read_csv(path, sep=sep, **kw)
    return df

print("== 1) 저장된 raw 파일 목록과 검증 ==")
targets = [
    ("data/raw/all-weeks-global.csv", ",", {}),
    ("data/raw/all-weeks-countries.csv", ",", {"encoding": "cp1252"}),
    ("data/raw/all-weeks-global_2026-11-11.tsv", "\t", {}),
    ("data/raw/all-weeks-countries_2026-11-11.tsv", "\t", {}),
]
for rel, sep, kw in targets:
    p = ROOT / rel
    df = inspect(p, sep, **kw)
    wc = [c for c in df.columns if "week" in c.lower()]
    period = f"{df[wc[0]].min()} ~ {df[wc[0]].max()}" if wc else "?"
    print(f"  {rel}")
    print(f"    행 {len(df):,} | 기간 {period} | 고유 작품 {df.show_title.nunique():,}편")

print("\n== 2) 결합 가능성 검증 (구형 끝 = 신형 시작과 이어지는지) ==")
g_old = inspect(ROOT / "data/raw/all-weeks-global.csv", ",")
g_new = inspect(ROOT / "data/raw/all-weeks-global_2026-11-11.tsv", "\t")
print(f"구형 마지막 주: {g_old.week.max()} / 신형 첫 주: {g_new.week.min()}")
gap_ok = pd.to_datetime(g_old.week.max()) + pd.Timedelta(days=7) == pd.to_datetime(g_new.week.min())
print("주차 연속성:", "OK — 결합 가능" if gap_ok else "확인 필요 (구형/신형 사이 빈 주차 발생)")

print("\n== 3) 결합 후 기대치 ==")
c_new = inspect(ROOT / "data/raw/all-weeks-countries_2026-11-11.tsv", "\t")
c_all = pd.concat([inspect(ROOT / "data/raw/all-weeks-countries.csv", ",", encoding="cp1252"), c_new]).drop_duplicates()
print(f"결합 countries: {len(c_all):,}행 | {c_all.week.nunique()}주 | {c_all.country_name.nunique()}개국 | "
      f"{c_all.show_title.nunique():,}편 — 281주·94개국·12,486편과 대조")


== 1) 저장된 raw 파일 목록과 검증 ==
  data/raw/all-weeks-global.csv
    행 1,160 | 기간 2021-07-04 ~ 2022-01-16 | 고유 작품 401편
  data/raw/all-weeks-countries.csv
    행 54,520 | 기간 2021-07-04 ~ 2022-01-16 | 고유 작품 2,006편
  data/raw/all-weeks-global_2026-11-11.tsv
    행 10,080 | 기간 2022-01-23 ~ 2026-11-15 | 고유 작품 3,319편
  data/raw/all-weeks-countries_2026-11-11.tsv
    행 468,840 | 기간 2022-01-23 ~ 2026-11-15 | 고유 작품 11,737편

== 2) 결합 가능성 검증 (구형 끝 = 신형 시작과 이어지는지) ==
구형 마지막 주: 2022-01-16 / 신형 첫 주: 2022-01-23
주차 연속성: OK — 결합 가능

== 3) 결합 후 기대치 ==


결합 countries: 523,360행 | 281주 | 94개국 | 12,486편 — 281주·94개국·12,486편과 대조


**원본 코드** — `scripts/14_raw_data_combine.py`

In [3]:
# -*- coding: utf-8 -*-
# ─────────────────────────────────────────────────────────────────────────────
# [용도] 14. 원본 데이터 결합·정리 — 구(2021-07~2022-01) + 신(2022-01~2026-11)
#        데이터를 결합해 분석 기준 테이블을 만든다
#        출력: data/interim/all-weeks-combined_global.csv / _countries.csv
# [환경] Python 3.15.2 (Windows) / pandas 3.1.1, numpy 2.5.4 — requirements.txt 참고
#        실행: python scripts/14_raw_data_combine.py
# [경로] ROOT = 스크립트 위치 기준 프로젝트 루트 (scripts/ 상위)
# [재현] 입력(raw 파일 4개)이 같으면 출력이 결정적으로 동일함 — 완전 재현 가능
#        ※ 하위 분석 스크립트(05~08)는 이 결합 과정을 내장하고 있어 독립 실행 가능
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")

# ---- 1) 구형 데이터 (2021-07-04 ~ 2022-01-16, 쉼표 구분 / 국가별 파일은 cp1252) ----
g_old = pd.read_csv(ROOT / "data/raw/all-weeks-global.csv", sep=",")
c_old = pd.read_csv(ROOT / "data/raw/all-weeks-countries.csv", sep=",", encoding="cp1252")
print(f"구형 global: {len(g_old)}행 ({g_old.week.min()} ~ {g_old.week.max()})")
print(f"구형 countries: {len(c_old)}행 ({c_old.week.min()} ~ {c_old.week.max()})")

# ---- 2) 신형 데이터 (2022-01-23 ~ 2026-11-15, 탭 구분 TSV) ----
g_new = pd.read_csv(ROOT / "data/raw/all-weeks-global_2026-11-11.tsv", sep="\t")
c_new = pd.read_csv(ROOT / "data/raw/all-weeks-countries_2026-11-11.tsv", sep="\t")
print(f"신형 global: {len(g_new)}행 ({g_new.week.min()} ~ {g_new.week.max()})")
print(f"신형 countries: {len(c_new)}행 ({c_new.week.min()} ~ {c_new.week.max()})")

# ---- 3) 결합 + 완전 중복 제거 ----
g_all = pd.concat([g_old, g_new]).drop_duplicates()
c_all = pd.concat([c_old, c_new]).drop_duplicates()
print(f"\n결합 global: {len(g_all)}행 | 고유 작품 {g_all.show_title.nunique()}편")
print(f"결합 countries: {len(c_all)}행 | 고유 작품 {c_all.show_title.nunique()}편 | 국가 {c_all.country_name.nunique()}개국")
print(f"분석 기간: {c_all.week.min()} ~ {c_all.week.max()} ({c_all.week.nunique()}주)")

# ---- 4) 저장 (중간 산출물 — data/interim) ----
g_all.to_csv(ROOT / "data/interim/all-weeks-combined_global.csv", index=False, encoding="utf-8-sig")
c_all.to_csv(ROOT / "data/interim/all-weeks-combined_countries.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료: data/interim/all-weeks-combined_global.csv, _countries.csv")

# ---- 5) 기대치 대조 ----
assert g_all.week.nunique() == 281, "주차 수 불일치!"
assert c_all.country_name.nunique() == 94, "국가 수 불일치!"
assert c_all.show_title.nunique() == 12486, "고유 작품 수 불일치!"
print("기대치 대조 통과: 281주 · 94개국 · 12,486편 ✓")


구형 global: 1160행 (2021-07-04 ~ 2022-01-16)
구형 countries: 54520행 (2021-07-04 ~ 2022-01-16)
신형 global: 10080행 (2022-01-23 ~ 2026-11-15)
신형 countries: 468840행 (2022-01-23 ~ 2026-11-15)



결합 global: 11240행 | 고유 작품 3585편
결합 countries: 523360행 | 고유 작품 12486편 | 국가 94개국
분석 기간: 2021-07-04 ~ 2026-11-15 (281주)



저장 완료: data/interim/all-weeks-combined_global.csv, _countries.csv
기대치 대조 통과: 281주 · 94개국 · 12,486편 ✓


### 인구 데이터 (UN WPP) — 수집 및 구조

인구 데이터는 UN World Population Prospects(OWID 정리본)에서 다운로드한 뒤, 94개국의 연도별 인구(2021~2026)를
`references/country_population_by_year.csv`로 정리했다. 국가 이름은 넷플릭스 데이터의 country_name과 맞춰 사용한다.

In [4]:
pop = pd.read_csv(ROOT / "references/country_population_by_year.csv")
print(f"인구 데이터: {len(pop)}개국 × 연도별 컬럼")
print(pop[["country_name", "continent", "2021", "2024"]].head(5).to_string(index=False))
print(f"\n대륙 분포: {pop['continent'].value_counts().to_dict()}")

인구 데이터: 94개국 × 연도별 컬럼
country_name continent        2021        2024
   Argentina        남미  45312282.0  45696153.0
     Bolivia        남미  11937369.0  12413318.0
      Brazil        남미 209550291.0 211998565.0
       Chile        남미  19456323.0  19764774.0
    Colombia        남미  51188174.0  52886360.0

대륙 분포: {'유럽': 34, '아시아': 25, '북중미': 15, '남미': 10, '아프리카': 7, '오세아니아': 3}


### 대륙 매핑 (references/country_continent.csv)

국가 이름(영문)을 대륙(유럽·아시아·북중미·남미·아프리카·오세아니아)에 매핑하는 테이블.
언어 교차 분석, 대륙별 확산 피처, 시장 파워 블록 구분의 기준이 된다.

In [5]:
cont = pd.read_csv(ROOT / "references/country_continent.csv")
print(f"대륙 매핑: {len(cont)}개국")
print(cont["대륙"].value_counts().to_string())

대륙 매핑: 94개국
대륙
유럽       34
아시아      25
북중미      15
남미       10
아프리카      7
오세아니아     3
